In [5]:
from sound_simulation import *
from beamforming import *
from sensor_array import *
from scene import *
from util import *
from psd_estimation import *
from plot_utils import *
from cropac import *
from ml_util import *

import pathlib
import numpy as np
import matplotlib.pyplot as plt

In [8]:
duration = 2
c = 344

N = WINDOW_LENGTH
step = WINDOW_STEP
window = WINDOW
fmin = FMIN
fmax = FMAX
VN, f, validf = NSTFT, VALID_FREQUENCIES, VALID_FREQUENCY_INDICES

sensor_array = array_circle8()

mvdr_estimator = MVDR_PSDE(db2lin(10))
cropac_estimator = CroPaC_PSDE(0)
nobec_estimator = NOBEC_PSDE(db2lin(10), db2lin(-20), order = 2)

estimators = [
    ["nobec", nobec_estimator],
    ["cropac", cropac_estimator],
    ["mvdr", mvdr_estimator],
]

paths = [str(path.relative_to(".").with_suffix("")) for path in pathlib.Path("../dataset/synthetic_training").rglob("*.wav")]

NE = len(estimators)

for path in paths:

    audio, sr = sf.read(path+".wav", always_2d=True)
    audio = audio.T
    with open(path+".json", "r", encoding="utf-8") as fil:
        target_udir = np.array(json.load(fil))

    k = 2 * np.pi * f / c

    SN, tidx, snaps = stft(audio, window, step, validf)
    target_steer = create_steering_vectors(create_distance_vectors(target_udir[None,:], sensor_array.positions), k[:,None])

    Sx = np.array([compute_covariance_matrices(snap) for snap in snaps])
    Sx = fb_average(Sx, sensor_array.positions)
    Sx = time_average(Sx, smoothing_alpha(sr*0.1, step, 0.5))

    params = PSDEParams(
        sensor_array.positions,
        target_udir, k,
        target_steer,
        compute_covariance_matrices(snaps[0]))

    for ei in range(NE):

        psd = np.array([estimators[ei][1].update_and_estimate(params, s) for s in Sx])
        
        feat_stft = extract_raw_stft(psd)
        feat_mfcc = extract_raw_mfcc(psd, sr, fmin, fmax, N, validf)

        with open(f"{path}_{estimators[ei][0]}_stft.npy", "wb") as fil:
            np.save(fil, feat_stft)
        with open(f"{path}_{estimators[ei][0]}_mfcc.npy", "wb") as fil:
            np.save(fil, feat_mfcc)

